# 05 - Optimisation, modèle final et suivi MLflow

In [1]:
import logging
import os
import warnings

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from xgboost import XGBClassifier

logging.getLogger("mlflow").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", module="mlflow")
sns.set_theme(style="whitegrid")

## Configuration et données

In [2]:
DATA_PATH = "data/processed/clean_eng.csv"
FIG_DIR = "reports/figures"
SEED = 42
MLFLOW_URI = "sqlite:///mlflow.db"
EXPERIMENT = "churn-prediction"
SCORING = ["accuracy", "precision", "recall", "f1", "roc_auc"]
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

NUMERIC = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
ENGINEERED = ["charges_per_tenure", "is_new_customer", "is_month_to_month", "is_fiber", "is_electronic_check", "nb_services"]
ORDINAL = {
    "Contract": ["Month-to-month", "One year", "Two year"],
    "MultipleLines": ["No phone service", "No", "Yes"],
    "InternetService": ["No", "DSL", "Fiber optic"],
}
NOMINAL = [
    "gender", "Partner", "Dependents", "PhoneService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "PaperlessBilling", "PaymentMethod",
]

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)

df = pd.read_csv(DATA_PATH)
y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["Churn", "customerID"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print("train:", X_train.shape, "test:", X_test.shape)
print("churn train / test:", round(y_train.mean(), 3), "/", round(y_test.mean(), 3))

train: (5634, 26) test: (1409, 26)
churn train / test: 0.265 / 0.265


### Preprocessing

In [3]:
def build_preprocessor(use_fe=True, use_cluster=False):
    num_cols = NUMERIC + (ENGINEERED if use_fe else [])
    nom_cols = NOMINAL + (["cluster"] if use_cluster else [])
    numeric = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    ordinal = make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OrdinalEncoder(categories=list(ORDINAL.values()), handle_unknown="use_encoded_value", unknown_value=-1),
    )
    nominal = make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore", drop="if_binary"))
    return ColumnTransformer(
        [("num", numeric, num_cols), ("ord", ordinal, list(ORDINAL)), ("nom", nominal, nom_cols)],
        verbose_feature_names_out=False,
    )

### Variable `cluster`

In [4]:
N_CLUSTERS = 3
CLUSTER_NUMERIC = ["SeniorCitizen", "tenure", "MonthlyCharges"] + ENGINEERED
CLUSTER_CATEGORICAL = list(ORDINAL) + NOMINAL + ["tenure_band"]


def build_cluster_pipeline(k=N_CLUSTERS):
    prep = ColumnTransformer(
        [
            ("num", StandardScaler(), CLUSTER_NUMERIC),
            ("cat", make_pipeline(OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), StandardScaler()), CLUSTER_CATEGORICAL),
        ]
    )
    return make_pipeline(prep, KMeans(n_clusters=k, n_init=10, random_state=SEED))


class ClusterFeature(BaseEstimator, TransformerMixin):
    def __init__(self, k=N_CLUSTERS):
        self.k = k

    def fit(self, X, y=None):
        self.model_ = build_cluster_pipeline(self.k).fit(X)
        return self

    def transform(self, X):
        out = X.copy()
        out["cluster"] = self.model_.predict(X)
        return out

### Pipeline et suivi MLflow

In [5]:
def make_model(clf, use_fe=True, use_cluster=False, sampler=None):
    steps = [("cluster", ClusterFeature())] if use_cluster else []
    steps.append(("prep", build_preprocessor(use_fe, use_cluster)))
    if sampler is not None:
        steps.append(("sampler", sampler))
    steps.append(("clf", clone(clf)))
    return Pipeline(steps)


def test_metrics(model, threshold=0.5):
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= threshold).astype(int)
    return {
        "test_accuracy": accuracy_score(y_test, pred),
        "test_precision": precision_score(y_test, pred),
        "test_recall": recall_score(y_test, pred),
        "test_f1": f1_score(y_test, pred),
        "test_roc_auc": roc_auc_score(y_test, proba),
    }


def run_experiment(name, clf, use_fe=True, use_cluster=False, sampler=None, threshold=0.5, tags=None):
    model = make_model(clf, use_fe, use_cluster, sampler)
    cv = cross_validate(model, X_train, y_train, cv=CV, scoring=SCORING, n_jobs=-1)
    metrics = {f"cv_{s}": cv[f"test_{s}"].mean() for s in SCORING}
    metrics["cv_roc_auc_std"] = cv["test_roc_auc"].std()
    model.fit(X_train, y_train)
    metrics.update(test_metrics(model, threshold))

    with mlflow.start_run(run_name=name):
        mlflow.set_tags({"model": type(clf).__name__, **(tags or {})})
        mlflow.log_params(
            {
                "use_feature_engineering": use_fe,
                "use_cluster": use_cluster,
                "sampler": type(sampler).__name__ if sampler is not None else "none",
                "scaler": "StandardScaler",
                "encoding": "OneHot(nominal) + Ordinal(Contract, MultipleLines, InternetService)",
                "imputation": "median(num) / most_frequent(cat)",
                "split": "80/20 stratified, seed 42",
                "threshold": threshold,
            }
        )
        mlflow.log_params({f"clf__{k}": v for k, v in clf.get_params().items()})
        mlflow.log_metrics(metrics)
        pred = (model.predict_proba(X_test)[:, 1] >= threshold).astype(int)
        cm = ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["Reste", "Churn"], cmap="Blues")
        mlflow.log_figure(cm.figure_, "confusion_matrix.png")
        roc = RocCurveDisplay.from_estimator(model, X_test, y_test)
        mlflow.log_figure(roc.figure_, "roc_curve.png")
        plt.close("all")
        mlflow.sklearn.log_model(model, name="model", input_example=X_test.head(3), serialization_format="cloudpickle")
    return model, metrics

In [6]:
import json

import joblib
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_predict

## 1. Optimisation des hyperparamètres

In [7]:
searches = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000, random_state=SEED),
        {"clf__C": [0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10]},
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=SEED),
        {
            "clf__n_estimators": [200, 400],
            "clf__max_depth": [4, 6, 8, 10, None],
            "clf__min_samples_leaf": [1, 5, 10, 20],
            "clf__max_features": ["sqrt", 0.5],
        },
    ),
    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=SEED),
        {
            "clf__n_estimators": [100, 200, 300],
            "clf__learning_rate": [0.02, 0.05, 0.1],
            "clf__max_depth": [2, 3, 4],
            "clf__subsample": [0.7, 0.85, 1.0],
        },
    ),
    "XGBoost": (
        XGBClassifier(random_state=SEED),
        {
            "clf__n_estimators": [200, 400],
            "clf__learning_rate": [0.01, 0.03, 0.1],
            "clf__max_depth": [2, 3, 4, 6],
            "clf__subsample": [0.7, 1.0],
            "clf__colsample_bytree": [0.7, 1.0],
            "clf__min_child_weight": [1, 5],
        },
    ),
}

best_params = {}
for name, (clf, grid) in searches.items():
    pipe = make_model(clf, sampler=SMOTE(random_state=SEED))
    if name == "Logistic Regression":
        search = GridSearchCV(pipe, grid, cv=CV, scoring="roc_auc", n_jobs=-1)
    else:
        search = RandomizedSearchCV(pipe, grid, n_iter=20, cv=CV, scoring="roc_auc", n_jobs=-1, random_state=SEED)
    search.fit(X_train, y_train)
    best_params[name] = {k.removeprefix("clf__"): v for k, v in search.best_params_.items()}
    print(f"{name:<20} CV ROC-AUC={search.best_score_:.4f}  {best_params[name]}")

Logistic Regression  CV ROC-AUC=0.8473  {'C': 0.1}


Random Forest        CV ROC-AUC=0.8450  {'n_estimators': 400, 'min_samples_leaf': 20, 'max_features': 'sqrt', 'max_depth': 10}


Gradient Boosting    CV ROC-AUC=0.8464  {'subsample': 0.7, 'n_estimators': 300, 'max_depth': 2, 'learning_rate': 0.05}


XGBoost              CV ROC-AUC=0.8463  {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.03, 'colsample_bytree': 1.0}
